## Install Dependencies

In [ ]:
# Install dependencies
%pip install -q amplpy matplotlib

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["gurobi", "xpress", "highs", "gokestrel"], # modules to install
    license_uuid="default") # license to use

### AMPL Formulation

In [7]:
%%ampl_eval
reset;                                          # Clear previous model

# ===== LITERAL SET =====
set SHIFTS = {"Day", "Evening", "Night"};      # Fixed set with explicit members
display SHIFTS;                                 # Output: Day, Evening, Night

# ===== ARITHMETIC RANGE SET =====
param start integer;                            # Lower bound parameter
param end > start integer;                      # Upper bound (must exceed start)
param interval > 0 integer;                     # Step size parameter
set YEARS = start .. end by interval;           # Computed range set

data;                                           # Supply parameter values
param start = 1;
param end = 10;
param interval = 1;
display YEARS;                                  # Output: 1,2,3,...,10

# ===== FILTERED COMPREHENSION SET =====
set PROD = {"coils", "plate", "rods"};          # Base product set
param cost {PROD} >= 0;                         # Cost per product
set CHEAP = {p in PROD: cost[p] < 10};          # Filter: products costing < 10

data;
param cost =                                     # Assign cost values
coils 5
plate 15
rods 25;
display CHEAP;                                  # Output: only coils (cost=5)

# ===== SET ALGEBRA COMBINATIONS =====
set PROD_A = {"bands", "coils"};                # First set
set PROD_B = {"coils", "plate", "rods"};        # Second set

set ALL_PROD   = PROD_A union PROD_B;            # Union: all items from both
set COMMON     = PROD_A inter PROD_B;            # Intersection: items in both
set ONLY_A     = PROD_A diff PROD_B;             # Difference: in A but not B
set EXCLUSIVE  = PROD_A symdiff PROD_B;          # Symmetric diff: in A or B not both
set PAIRS      = PROD_A cross PROD_B;            # Cartesian product: all (a,b) pairs

display ALL_PROD, COMMON, ONLY_A, EXCLUSIVE, PAIRS;  # Output all 5 results

set SHIFTS := Day Evening Night;

set YEARS := 1 2 3 4 5 6 7 8 9 10;

set CHEAP := coils;

set ALL_PROD := bands coils plate rods;

set COMMON := coils;

set ONLY_A := bands;

set EXCLUSIVE := bands plate rods;

set PAIRS :=
(bands,coils)   (bands,rods)    (coils,plate)
(bands,plate)   (coils,coils)   (coils,rods);



### amplpy Formulation

In [14]:
%%ampl_eval
reset;

# Literal set — fixed members embedded in the model
set SHIFTS;

# Arithmetic range parameterized by data — recomputed if start/end/interval change
param start =1;
param end = 10;
param interval = 1;
set YEARS = start .. end by interval;

# Filtered comprehension — CHEAP depends on PROD and cost; updated automatically
set PROD;
param cost {PROD} >= 0;
set CHEAP = {p in PROD: cost[p] < 10};

# set-algebraic combination
set PROD_A;
set PROD_B;

set ALL_PROD   = PROD_A union PROD_B;          # {"bands", "coils", "plate", "rods"}
set COMMON     = PROD_A inter PROD_B;          # {"coils"}
set ONLY_A     = PROD_A diff PROD_B;           # {"bands"}
set EXCLUSIVE  = PROD_A symdiff PROD_B;        # {"bands", "plate", "rods"}
set PAIRS      = PROD_A cross PROD_B;          # all ordered pairs (a, b)

In [16]:
# Set 1: Literal set from Python list
shifts = ["Day", "Evening", "Night"]
ampl.set["SHIFTS"].set_values(shifts)

# Set 3: Filtered set CHEAP based on cost
prod = ["coils", "plate", "rods"]
cost_data = {"coils": 5, "plate": 15, "rods": 25}

ampl.set["PROD"].set_values(prod)
ampl.param["cost"].set_values(cost_data)

# Set 4: Set-algebraic combinations
prod_a = ["bands", "coils"]
prod_b = ["coils", "plate", "rods"]

ampl.set["PROD_A"].set_values(prod_a)
ampl.set["PROD_B"].set_values(prod_b)

print(f"SHIFTS:      {ampl.set['SHIFTS'].getValues()} (size: {ampl.set['SHIFTS'].size()})")
print(f"YEARS:       {ampl.set['YEARS'].getValues()}")
print(f"PROD:        {ampl.set['PROD'].getValues()} (size: {ampl.set['PROD'].size()})")
print(f"CHEAP:       {ampl.set['CHEAP'].getValues()} (size: {ampl.set['CHEAP'].size()})")
print(f"ALL_PROD:    {ampl.set['ALL_PROD'].getValues()}")
print(f"COMMON:      {ampl.set['COMMON'].getValues()}")
print(f"ONLY_A:      {ampl.set['ONLY_A'].getValues()}")
print(f"EXCLUSIVE:   {ampl.set['EXCLUSIVE'].getValues()}")
print(f"PAIRS:       {ampl.set['PAIRS'].getValues()}")
print(f"Costs:       {dict(ampl.param['cost'].get_values())}")

SHIFTS:         SHIFTS   
   'Day'       
 'Evening'     
  'Night'      
 (size: 3)
YEARS:          YEARS    
     1         
     2         
     3         
     4         
     5         
     6         
     7         
     8         
     9         
     10        

PROD:            PROD    
  'coils'      
  'plate'      
   'rods'      
 (size: 3)
CHEAP:          CHEAP    
  'coils'      
 (size: 1)
ALL_PROD:      ALL_PROD  
  'bands'      
  'coils'      
  'plate'      
   'rods'      

COMMON:         COMMON   
  'coils'      

ONLY_A:         ONLY_A   
  'bands'      

EXCLUSIVE:    EXCLUSIVE  
  'bands'      
  'plate'      
   'rods'      

PAIRS:          index0       index1   
  'bands'        'coils'      
  'bands'        'plate'      
  'bands'         'rods'      
  'coils'        'coils'      
  'coils'        'plate'      
  'coils'         'rods'      

Costs:       {'coils': 5, 'plate': 15, 'rods': 25}
